# Perch AUC on the training clips

Each clip in `train_audio` is scored by Perch on every 5 s window (1 s stride); a class's clip score is the max or
the mean of its window scores. A class is positive for a clip if it is the clip's primary label or one of its
secondary labels. Window scores are compared in three forms:

- **raw** logits;
- **normalized** logits: per label, the mean over every window (clips and soundscapes) maps to 0, and the 99th
  percentile over the windows of its positive clips maps to 1 (`norm_stats`);
- **softmax** over our 203 labels in each window.

**Macro** AUC averages the per-label ROC-AUC over the labels with at least one positive and one negative clip.
**Micro** AUC treats every (clip, label) pair as one sample, so it also depends on whether scores are comparable
across labels. The last column keeps only the labels whose raw per-label AUC is above 0.9; that selection is made on
the same clips, so it is optimistic.

Perch v2 was trained on Xeno-canto and iNaturalist recordings, the same sources as these clips, so some of them may
be in its training data: none of these numbers is a held-out score.

In [ ]:
import ast
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from birdclef import config as C
from birdclef.perch import norm_stats
from birdclef.utils import macro_auc

THRESHOLD = 0.9

In [ ]:
perch_map = pd.read_csv(C.PERCH_DIR / "perch_map.csv").astype({"primary_label": str})
classes = perch_map.primary_label.tolist()
column = {c: i for i, c in enumerate(classes)}

train = pd.read_csv(C.RAW_DIR / "train.csv").astype({"primary_label": str})
train["npz"] = [(C.PERCH_DIR / "train_audio" / f).with_suffix(".npz") for f in train.filename]
train = train[[p.exists() for p in train.npz]].reset_index(drop=True)
train["labels"] = [[p, *ast.literal_eval(s)] for p, s in zip(train.primary_label, train.secondary_labels)]

y = np.zeros((len(train), len(classes)), np.int8)
for i, labels in enumerate(train.labels):
    y[i, [column[c] for c in labels if c in column]] = 1
print(f"{len(train)} clips, {len(classes)} Perch classes, {y.sum()} positive labels")

In [ ]:
def softmax(x):
    z = np.exp(x - x.max(1, keepdims=True))
    return z / z.sum(1, keepdims=True)


MEAN, TOP, fallback = norm_stats(classes)
print(f"{len(fallback)} labels without a positive clip use the soundscape fallback")
transforms = {
    "raw": lambda x: x,
    "normalized": lambda x: (x - MEAN) / np.maximum(TOP - MEAN, 1e-6),
    "softmax": softmax,
}
scores = {(t, a): np.empty(y.shape, np.float32) for t in transforms for a in ("max", "mean")}
for i, path in enumerate(train.npz):
    logits = np.load(path)["logits"].astype(np.float32)
    for t, f in transforms.items():
        x = f(logits)
        scores[t, "max"][i], scores[t, "mean"][i] = x.max(0), x.mean(0)

In [ ]:
rows = {}
for (t, a), s in scores.items():
    macro, per_class = macro_auc(y, s)
    if t == "raw":
        perch_map[f"auc_{a}"] = per_class
    keep = (perch_map[f"auc_{a}"] > THRESHOLD).to_numpy()
    rows[f"{t}, {a}"] = {
        "macro": macro,
        "micro": roc_auc_score(y.ravel(), s.ravel()),
        f"micro, labels > {THRESHOLD}": roc_auc_score(y[:, keep].ravel(), s[:, keep].ravel()) if keep.any() else np.nan,
    }
print(f"{perch_map.auc_max.notna().sum()} labels scored, {(perch_map.auc_max > THRESHOLD).sum()} above {THRESHOLD}")
table = pd.DataFrame(rows).T.round(4)
table.to_csv("results/perch_auc_clips.csv")
table

In [ ]:
scored = perch_map.dropna(subset="auc_max").sort_values("auc_max").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(13, 4))
for k, g in enumerate(sorted(scored.class_name.unique())):
    part = scored[scored.class_name == g]
    ax.bar(part.index, part.auc_max, width=1.0, color=plt.cm.tab10(k), label=f"{g} ({len(part)})")
ax.axhline(scored.auc_max.mean(), color="black", lw=0.8, ls="--", label=f"macro AUC {scored.auc_max.mean():.4f}")
ax.set_xlim(-0.5, len(scored) - 0.5)
ax.set_ylim(max(0, scored.auc_max.min() - 0.05), 1)
ax.set_xticks([])
ax.set_xlabel("label, sorted by AUC")
ax.set_ylabel("ROC-AUC (raw logits, max over windows)")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig("results/perch_auc_clips.png", dpi=110, bbox_inches="tight")
plt.show()
print(scored.head(10)[["primary_label", "scientific_name", "class_name", "auc_max", "auc_mean"]].to_string(index=False))